# Notebook 08 – Personality Nudges

Generate **lightweight personality refinements** for styling adjustments.

Personality **never replaces the interface** — it only nudges configurable properties such as:

- Visual Richness
- Information Density
- Whitespace
- Animation
- Recommendation Strength

For every Big Five trait level (Low / Medium / High), survey tendencies supported by Notebook 03 and Notebook 04 evidence are converted into ordinal nudges (`-1`, `+1`).


## Inputs
- `data/processed/clean_dataset.csv`
- Notebook 03 statistical results
- Notebook 04 Random Forest + SHAP

## Outputs
- `data/outputs/trait_modifiers.json`
- `reports/PersonalityNudges/trait_modifiers.xlsx`


In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.personality_nudges.repository import run_personality_nudge_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("PersonalityNudges")
OUTPUT_DIR = PATHS.data_outputs

print(f"Reports: {REPORTS}")
print(f"JSON output: {OUTPUT_DIR}")


Reports: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/PersonalityNudges
JSON output: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs


## Check Inputs


In [2]:
INPUTS = {
    "Clean dataset": PATHS.data_processed / "clean_dataset.csv",
    "Statistical results (NB03)": PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.xlsx",
    "Feature importance (NB04)": PATHS.reports / "Feature_Importance" / "feature_importance.xlsx",
    "SHAP summary (NB04)": PATHS.reports / "Feature_Importance" / "shap_summary.csv",
}

for label, path in INPUTS.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'}")


Clean dataset: OK
Statistical results (NB03): OK
Feature importance (NB04): OK
SHAP summary (NB04): OK


## Generate Evidence-Backed Personality Nudges

1. Derive ordinal tendencies from the survey for each trait × level × property
2. Score with statistical + ML evidence (Notebook 03–04)
3. Export only nudges meeting the evidence threshold


In [3]:
result = run_personality_nudge_pipeline(PROJECT_ROOT, OUTPUT_DIR, REPORTS)

entries = result.entries
scored = result.scored_modifiers
summary = result.summary

display(scored.head(15) if not scored.empty else scored)


INFO: Personality nudges: 9 entries, 13 total nudges


,Trait,Level,Property,Nudge,Direction,Delta,Group_N,Baseline_Score,Group_Score,Provenance,Cramers_V,RF_Importance,Mean_SHAP,Effect_Size,Coverage,Modifier_Evidence_Score,Strength
0,Agreeableness,High,recommendation_emphasis,1,increase,NaN,94,0.7275,0.7287,theory,0.135870,0.135030,0.018411,0.0000,0.470,0.326507,Moderate
1,Agreeableness,Low,recommendation_emphasis,1,increase,0.0850,24,0.7275,0.8125,data-driven,0.135870,0.135030,0.018411,0.0850,0.120,0.633345,Strong
2,Conscientiousness,High,information_density,1,increase,NaN,66,0.5350,0.5227,theory,0.067581,0.136669,0.027815,0.0000,0.330,0.272673,Weak
3,Conscientiousness,Low,information_density,-1,decrease,NaN,35,0.5350,0.5500,theory,0.067581,0.136669,0.027815,0.0000,0.175,0.258723,Weak
4,Extraversion,High,recommendation_emphasis,-1,decrease,-0.0723,29,0.7275,0.6552,data-driven,0.147525,0.118417,0.025219,0.0723,0.145,0.635308,Strong
5,Extraversion,High,information_density,-1,decrease,-0.0522,29,0.5350,0.4828,data-driven,0.092127,0.149454,0.029968,0.0522,0.145,0.578200,Strong
6,Extraversion,High,animation_level,1,increase,NaN,29,NaN,NaN,theory,0.000000,0.000000,0.000000,0.0000,0.145,0.013050,Weak
7,Extraversion,Low,recommendation_emphasis,-1,decrease,NaN,61,0.7275,0.7131,theory,0.147525,0.118417,0.025219,0.0000,0.305,0.323285,Moderate
8,Neuroticism,High,recommendation_emphasis,1,increase,0.0503,45,0.7275,0.7778,data-driven,0.193360,0.160564,0.031520,0.0503,0.225,0.765863,Very Strong
9,Neuroticism,High,information_density,-1,decrease,NaN,45,0.5350,0.5556,theory,0.130845,0.159458,0.036613,0.0000,0.225,0.361396,Moderate


## Sample Nudge JSON


In [4]:
if entries:
    print(json.dumps(entries[0], indent=2, ensure_ascii=False))


{
  "trait": "Extraversion",
  "level": "Low",
  "nudges": {
    "recommendation_strength": -1
  },
  "confidence": 0.323285
}


## Modifiers per Trait


In [5]:
for trait, count in summary["modifiers_per_trait"].items():
    print(f"{trait}: {count}")


Extraversion: 3
Agreeableness: 2
Conscientiousness: 2
Neuroticism: 3
Openness: 3


## Exports


In [6]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")


Export locations:
- trait_modifiers_json: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/data/outputs/trait_modifiers.json
- trait_modifiers_xlsx: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/trait_modifiers.xlsx
- trait_modifiers_csv: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/trait_modifiers.csv
- summary_md: /Users/mariam/Downloads/Trial-Guide/EvidenceBasedAdaptiveUI/reports/PersonalityNudges/personality_nudges_summary.md


## Final Output


In [7]:
print("Modifiers per trait")
for trait, count in summary["modifiers_per_trait"].items():
    print(f"  {trait}: {count}")
print("Repository generated.")


Modifiers per trait
  Extraversion: 3
  Agreeableness: 2
  Conscientiousness: 2
  Neuroticism: 3
  Openness: 3
Repository generated.
